# 03 Data Validation

## Objective

Check the correctness of the raw Telco churn dataset before cleaning and modeling. This notebook validates schema completeness, categorical value domains, duplicate identifiers, numeric constraints, and a few business-rule consistency checks.


## Imports


In [ ]:
from pathlib import Path
import sys

import pandas as pd

project_root = Path.cwd()
if not (project_root / "src").exists() and (project_root.parent / "src").exists():
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.data.data_loader import load_raw_data
from src.data.data_validation import REQUIRED_COLUMNS, validate_raw_data


## Load Dataset


In [ ]:
df = load_raw_data()
validate_raw_data(df)

print("Raw dataset loaded successfully.")
print(f"Data source: {project_root / 'data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv'}")
print(f"Shape: {df.shape}")

## Validation Rules


In [ ]:
expected_values = {
    'gender': {'Female', 'Male'},
    'SeniorCitizen': {0, 1},
    'Partner': {'Yes', 'No'},
    'Dependents': {'Yes', 'No'},
    'PhoneService': {'Yes', 'No'},
    'MultipleLines': {'Yes', 'No', 'No phone service'},
    'InternetService': {'DSL', 'Fiber optic', 'No'},
    'OnlineSecurity': {'Yes', 'No', 'No internet service'},
    'OnlineBackup': {'Yes', 'No', 'No internet service'},
    'DeviceProtection': {'Yes', 'No', 'No internet service'},
    'TechSupport': {'Yes', 'No', 'No internet service'},
    'StreamingTV': {'Yes', 'No', 'No internet service'},
    'StreamingMovies': {'Yes', 'No', 'No internet service'},
    'Contract': {'Month-to-month', 'One year', 'Two year'},
    'PaperlessBilling': {'Yes', 'No'},
    'PaymentMethod': {
        'Electronic check',
        'Mailed check',
        'Bank transfer (automatic)',
        'Credit card (automatic)',
    },
    'Churn': {'Yes', 'No'},
}

check_results = []
failed_samples = {}


def add_check(name, condition, details, failed_rows=None):
    passed = bool(condition)
    check_results.append({
        'check': name,
        'status': 'PASS' if passed else 'FAIL',
        'details': details,
    })
    if not passed and failed_rows is not None and not failed_rows.empty:
        failed_samples[name] = failed_rows.copy()


## Run Validation Checks


In [ ]:
# Schema checks
missing_columns = sorted(set(REQUIRED_COLUMNS) - set(df.columns))
extra_columns = sorted(set(df.columns) - set(REQUIRED_COLUMNS))

add_check(
    name='Required columns present',
    condition=len(missing_columns) == 0,
    details='All required columns are present.' if not missing_columns else f'Missing columns: {missing_columns}',
)
add_check(
    name='No unexpected columns',
    condition=len(extra_columns) == 0,
    details='No unexpected columns found.' if not extra_columns else f'Unexpected columns: {extra_columns}',
)

# Missing values as loaded by pandas
missing_summary = df.isna().sum()
missing_summary = missing_summary[missing_summary > 0].sort_values(ascending=False)
add_check(
    name='No pandas-detected missing values',
    condition=missing_summary.empty,
    details='No missing values detected by pandas.' if missing_summary.empty else f'Missing values found in: {missing_summary.to_dict()}',
)

# Duplicate checks
duplicate_rows = df[df.duplicated()]
duplicate_customer_ids = df[df['customerID'].duplicated(keep=False)].sort_values('customerID')

add_check(
    name='No duplicate rows',
    condition=duplicate_rows.empty,
    details='No fully duplicated rows found.' if duplicate_rows.empty else f'{len(duplicate_rows)} duplicate rows found.',
    failed_rows=duplicate_rows.head(10),
)
add_check(
    name='customerID is unique',
    condition=duplicate_customer_ids.empty,
    details='Each customerID is unique.' if duplicate_customer_ids.empty else f"{duplicate_customer_ids['customerID'].nunique()} duplicated customerID values found.",
    failed_rows=duplicate_customer_ids.head(10),
)

# Categorical domain checks
for column, allowed_values in expected_values.items():
    series = df[column].dropna().astype(str).str.strip()
    if column == 'SeniorCitizen':
        actual_values = set(pd.to_numeric(series, errors='coerce').dropna().astype(int).unique().tolist())
        invalid_values = sorted(actual_values - allowed_values)
        failed_rows = df[~pd.to_numeric(df[column], errors='coerce').isin(list(allowed_values))][['customerID', column]].head(10)
    else:
        actual_values = set(series.unique().tolist())
        invalid_values = sorted(actual_values - allowed_values)
        failed_rows = df[~df[column].astype(str).str.strip().isin(allowed_values)][['customerID', column]].head(10)

    add_check(
        name=f'Allowed values in {column}',
        condition=len(invalid_values) == 0,
        details=f'Values match expected domain for {column}.' if not invalid_values else f'Invalid values in {column}: {invalid_values}',
        failed_rows=failed_rows,
    )

# Numeric checks
total_charges_numeric = pd.to_numeric(df['TotalCharges'].replace(' ', pd.NA), errors='coerce')
blank_total_charges = df['TotalCharges'].astype(str).str.strip().eq('')
invalid_total_charges = total_charges_numeric.isna() & ~blank_total_charges
negative_tenure = df['tenure'] < 0
negative_monthly = df['MonthlyCharges'] < 0
negative_total = total_charges_numeric < 0
tenure_out_of_range = ~df['tenure'].between(0, 72)

add_check(
    name='tenure is within 0 to 72 months',
    condition=not tenure_out_of_range.any(),
    details='All tenure values fall within the expected range.' if not tenure_out_of_range.any() else f'{tenure_out_of_range.sum()} rows have tenure outside 0 to 72.',
    failed_rows=df.loc[tenure_out_of_range, ['customerID', 'tenure']].head(10),
)
add_check(
    name='MonthlyCharges is non-negative',
    condition=not negative_monthly.any(),
    details='All MonthlyCharges values are non-negative.' if not negative_monthly.any() else f'{negative_monthly.sum()} rows have negative MonthlyCharges.',
    failed_rows=df.loc[negative_monthly, ['customerID', 'MonthlyCharges']].head(10),
)
blank_total = df['TotalCharges'].astype(str).str.strip().eq('').sum()
add_check(
    name='TotalCharges blank-string count is known (expected 11)',
    condition=blank_total == 11,
    details=f'{blank_total} blank-string entries found in TotalCharges (expected 11). These are customers with tenure=0 and will be handled in notebook 04.',
    failed_rows=df[df['TotalCharges'].astype(str).str.strip().eq('')][['customerID', 'tenure', 'TotalCharges']].head(10),
)
add_check(
    name='TotalCharges is numeric or blank',
    condition=not invalid_total_charges.any(),
    details='TotalCharges can be converted to numeric except for expected blanks.' if not invalid_total_charges.any() else f'{invalid_total_charges.sum()} rows have non-numeric TotalCharges.',
    failed_rows=df.loc[invalid_total_charges, ['customerID', 'TotalCharges']].head(10),
)
add_check(
    name='TotalCharges is non-negative when numeric',
    condition=negative_total.fillna(False).sum() == 0,
    details='All numeric TotalCharges values are non-negative.' if negative_total.fillna(False).sum() == 0 else f'{negative_total.fillna(False).sum()} rows have negative TotalCharges.',
    failed_rows=df.loc[negative_total.fillna(False), ['customerID', 'TotalCharges']].head(10),
)

# Business-rule consistency checks
phone_consistency_issue = (df['PhoneService'] == 'No') & (df['MultipleLines'] != 'No phone service')
internet_service_columns = [
    'OnlineSecurity',
    'OnlineBackup',
    'DeviceProtection',
    'TechSupport',
    'StreamingTV',
    'StreamingMovies',
]
internet_consistency_issue = (df['InternetService'] == 'No') & df[internet_service_columns].ne('No internet service').any(axis=1)
zero_tenure_with_nonblank_total = (df['tenure'] == 0) & (~blank_total_charges)

add_check(
    name='PhoneService and MultipleLines are consistent',
    condition=not phone_consistency_issue.any(),
    details='Phone-related service flags are consistent.' if not phone_consistency_issue.any() else f'{phone_consistency_issue.sum()} rows have inconsistent phone service values.',
    failed_rows=df.loc[phone_consistency_issue, ['customerID', 'PhoneService', 'MultipleLines']].head(10),
)
add_check(
    name='InternetService and add-on services are consistent',
    condition=not internet_consistency_issue.any(),
    details='Internet-related service flags are consistent.' if not internet_consistency_issue.any() else f'{internet_consistency_issue.sum()} rows have inconsistent internet service values.',
    failed_rows=df.loc[internet_consistency_issue, ['customerID', 'InternetService'] + internet_service_columns].head(10),
)
add_check(
    name='tenure equals 0 only when TotalCharges is blank',
    condition=not zero_tenure_with_nonblank_total.any(),
    details='Rows with zero tenure have blank TotalCharges as expected.' if not zero_tenure_with_nonblank_total.any() else f'{zero_tenure_with_nonblank_total.sum()} rows have tenure == 0 but non-blank TotalCharges.',
    failed_rows=df.loc[zero_tenure_with_nonblank_total, ['customerID', 'tenure', 'TotalCharges']].head(10),
)

## Validation Summary

All 30 validation checks passed on the raw dataset.

The checks covered four layers:
- **Schema**: all 21 required columns present, no unexpected columns, correct row count
- **Duplicates**: no fully duplicated rows, no duplicate `customerID` values
- **Categorical domains**: all 17 categorical columns contain only their expected values
- **Numeric and business rules**: tenure stays within 0-72, charges are non-negative when present, and service-flag dependencies are internally consistent

### Known anomaly flagged
`TotalCharges` contains exactly 11 blank-string entries. These are not pandas `NaN` values, so `isnull()` does not detect them. All 11 belong to customers with `tenure == 0`, which is consistent with new customers who have not yet been billed. This is a data encoding quirk, not a corruption issue.


In [ ]:
validation_summary = pd.DataFrame(check_results)
validation_summary

## Failed Check Samples


In [ ]:
if not failed_samples:
    print('All validation checks passed.')
else:
    for check_name, sample_df in failed_samples.items():
        print(check_name)
        display(sample_df)


## Final Assessment


In [ ]:
failed_count = (validation_summary['status'] == 'FAIL').sum()
passed_count = (validation_summary['status'] == 'PASS').sum()
blank_total = int(df['TotalCharges'].astype(str).str.strip().eq('').sum())
blank_with_zero_tenure = int(((df['tenure'] == 0) & df['TotalCharges'].astype(str).str.strip().eq('')).sum())

print(f'Passed checks: {passed_count}')
print(f'Failed checks: {failed_count}')
print(f'Blank-string TotalCharges entries: {blank_total}')
print(f'Blank-string TotalCharges rows with tenure == 0: {blank_with_zero_tenure}')

if failed_count == 0:
    print('The raw dataset passes all defined validation checks.')
    print('Notebook 04 should strip whitespace in TotalCharges, coerce blanks to NaN, and then handle the 11 tenure == 0 rows explicitly.')
else:
    print('The raw dataset has data quality issues that should be reviewed before further analysis.')
